# UC2 — Schema enforcement + multi-version from 2 assets

**Persona:** data ingestion pipeline owner enforcing input contracts.

**Goal:**
1. PATCH `items_schema` to declare `code` and `name` as mandatory.
2. Ingest a feature missing `code` → expect **HTTP 207 Multi-Status**
   with an `IngestionReport.rejections[*]` describing the missing field
   and a `policy_source` URL pointing back to the schema config.
3. Ingest features from **two different assets** with the same `code`
   value; the `ItemsWritePolicy.on_conflict=new_version` plus
   `enable_validity=true` create a new version row; the previous row's
   `valid_to` is bounded.


In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:8])
CATALOG_ID = os.environ.get("CATALOG_ID", f"cf_uc2_{RUN_ID}")
COLLECTION_ID = os.environ.get("COLLECTION_ID", f"col_{RUN_ID}")

IS_LOCAL = "localhost" in BASE_URL or "127.0.0.1" in BASE_URL

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)

print(f"BASE_URL      : {BASE_URL}")
print(f"CATALOG_ID    : {CATALOG_ID}")
print(f"COLLECTION_ID : {COLLECTION_ID}")
print(f"AUTH          : {'token set' if TOKEN else 'anonymous'}")
if not TOKEN:
    print("\nWARNING: no Bearer token set — config writes will 401.")
    print("Set DYNASTORE_TOKEN before running write cells.")


In [ ]:
# Create catalog (idempotent: 409 = already exists)
catalog_payload = {
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": f"Cycle F UC walkthrough {RUN_ID}",
    "description": "Ephemeral catalog for cycle_f_use_cases notebook.",
    "stac_version": "1.0.0",
}
r = client.post("/stac/catalogs", content=json.dumps(catalog_payload))
if r.status_code in (200, 201):
    print(f"Catalog '{CATALOG_ID}' created.")
elif r.status_code == 409:
    print(f"Catalog '{CATALOG_ID}' already exists.")
else:
    raise RuntimeError(f"Catalog create failed: {r.status_code}: {r.text}")

# Create collection (idempotent)
collection_payload = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": f"UC collection {RUN_ID}",
    "description": "Walkthrough collection — defaults inherited then PATCHed.",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
    "links": [],
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection_payload),
)
if r.status_code in (200, 201):
    print(f"Collection '{COLLECTION_ID}' created.")
elif r.status_code == 409:
    print(f"Collection '{COLLECTION_ID}' already exists.")
else:
    raise RuntimeError(f"Collection create failed: {r.status_code}: {r.text}")


In [ ]:
# Helper — show explicit + waterfall-resolved view for a plugin.
#
# The configs API exposes:
#   GET /configs/.../plugins/{plugin_id}                  → explicit (per-scope)
#   GET /configs/.../plugins/{plugin_id}?resolved=true    → waterfall-resolved
# There is NO `/effective` endpoint — the resolved form is reached via
# the `?resolved=true` query string.
def show_config_delta(plugin_id: str, level: str = "collection") -> None:
    if level == "collection":
        base = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
    elif level == "catalog":
        base = f"/configs/catalogs/{CATALOG_ID}"
    else:
        raise ValueError(level)
    rx = client.get(f"{base}/plugins/{plugin_id}")
    rr = client.get(f"{base}/plugins/{plugin_id}", params={"resolved": "true"})
    print(f"\n=== {plugin_id} @ {level} ===")
    print("EXPLICIT (only fields stored at this scope):")
    if rx.status_code == 200:
        print(json.dumps(rx.json(), indent=2)[:600])
    else:
        print(f"  ({rx.status_code} — none stored, every field inherited)")
    print("RESOLVED (waterfall over platform → catalog → collection):")
    if rr.status_code == 200:
        print(json.dumps(rr.json(), indent=2)[:800])
    else:
        print(f"  ({rr.status_code}) {rr.text[:160]}")


In [ ]:
def put_config(plugin_id: str, body: dict, level: str = "collection") -> None:
    if level == "collection":
        url = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/{plugin_id}"
    else:
        url = f"/configs/catalogs/{CATALOG_ID}/plugins/{plugin_id}"
    r = client.put(url, content=json.dumps(body), timeout=60.0)
    print(f"PUT {plugin_id}: {r.status_code}")
    if r.status_code not in (200, 201, 204):
        print(f"  body: {r.text[:300]}")
        if r.status_code == 401:
            raise RuntimeError("Unauthorized — set DYNASTORE_TOKEN.")
        raise RuntimeError(f"PUT failed: {r.status_code}")


## Step 1 — PATCH driver with attributes sidecar (storage layout only)

Sidecars must be configured before any rows; same immutability rule as UC1. The behavioural knob `external_id_field` lives on `items_write_policy` (step 3), not on the sidecar.


In [ ]:
items_pg_driver = {
    "engine_ref": "postgresql_engine_config",
    "sidecars": [
        {"sidecar_type": "geometries"},
        {
            "sidecar_type": "attributes",
            "enable_external_id": True,
            "index_external_id": True,
            "enable_asset_id": True,
            "enable_validity": True,
        },
    ],
}
put_config("items_postgresql_driver_config", items_pg_driver)


## Step 2 — PATCH `items_schema` with mandatory fields

In [ ]:
schema_patch = {
    "fields": [
        {"name": "code", "type": "text", "required": True},
        {"name": "name", "type": "text", "required": True},
        {"name": "description", "type": "text"},
    ],
    "strict_unknown_fields": False,
}
put_config("items_schema", schema_patch)
show_config_delta("items_schema")


## Step 3 — PATCH `items_write_policy` for multi-version

In [ ]:
write_policy = {
    "on_conflict": "new_version",
    "identity_matchers": ["external_id"],
    "external_id_field": "properties.code",
    "track_asset_id": True,
    "enable_validity": True,
}
put_config("items_write_policy", write_policy)


## Step 4 — Ingest a feature missing `code` → 207 Multi-Status

Look at the `rejections` array — each entry includes `reason`,
`message`, `matcher` (which sidecar matcher fired), and a
`policy_source` URL pointing back to the schema config.


In [ ]:
bad = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": f"bad-{RUN_ID}",
    "collection": COLLECTION_ID,
    "geometry": {"type": "Point", "coordinates": [12.5, 41.9]},
    "bbox": [12.5, 41.9, 12.5, 41.9],
    "properties": {
        "datetime": "2024-01-10T00:00:00Z",
        # NB: NO "code" — should be rejected
        "name": "Missing code field",
    },
    "assets": {},
    "links": [],
}
ingest_url = f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"
r = client.post(ingest_url, content=json.dumps(bad))
print(f"POST bad feature: {r.status_code}")
if r.status_code == 207:
    report = r.json()
    print("rejections:")
    for rej in report.get("rejections", []):
        print(f"  - reason={rej.get('reason')!r} matcher={rej.get('matcher')!r}")
        print(f"    message={rej.get('message')!r}")
        print(f"    policy_source={rej.get('policy_source')!r}")
elif r.status_code in (400, 422):
    print(f"  body: {r.text[:400]}")
    print("  (single-feature ingest may surface 400/422 instead of 207)")
else:
    print(f"  unexpected: {r.text[:300]}")


## Step 5 — Register 2 assets and ingest features tagged with `code="K42"`

Asset registration goes through `POST /assets/catalogs/{cat}/collections/{coll}`
with a JSON body — no multipart upload.  In a full pipeline the binary
lives in GCS and gets registered automatically via OBJECT_FINALIZE
events; here we register the asset rows directly so the items can
carry the `asset_id` reference.  See
`notebook_showcase/notebooks/ui_walkthrough/02_upload_with_reporter.ipynb`
for the full asset → ingestion-process flow.

Each asset contributes one feature with `properties.code = "K42"`.
The first ingest creates the row; the second triggers the
`new_version` policy with the second asset's id and a bounded
`valid_to` on the first version.


In [ ]:
def register_asset(asset_id: str) -> str:
    r = client.post(
        f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
        content=json.dumps({
            "asset_id": asset_id,
            "asset_type": "ASSET",
            "uri": f"file:///fixtures/{asset_id}.geojson",  # placeholder
            "metadata": {"source": f"uc2-{RUN_ID}"},
        }),
    )
    print(f"  asset register {asset_id}: HTTP {r.status_code}")
    assert r.status_code in (200, 201, 409), f"asset register: {r.text[:200]}"
    return asset_id

asset_a = register_asset("pack-A")
asset_b = register_asset("pack-B")

def ingest_with_asset(asset_id: str, idx: int) -> int:
    feat = {
        "type": "Feature",
        "stac_version": "1.0.0",
        "id": f"k42-via-{asset_id}-{RUN_ID}",
        "collection": COLLECTION_ID,
        "geometry": {"type": "Point", "coordinates": [12.5 + 0.01*idx, 41.9]},
        "bbox": [12.5 + 0.01*idx, 41.9, 12.5 + 0.01*idx, 41.9],
        "properties": {
            "datetime": "2024-01-10T00:00:00Z",
            "code": "K42",
            "name": f"From {asset_id}",
            "asset_id": asset_id,
        },
        "assets": {},
        "links": [],
    }
    r = client.post(ingest_url, content=json.dumps(feat))
    print(f"  ingest via {asset_id}: {r.status_code}")
    return r.status_code

s1 = ingest_with_asset(asset_a, 0)
s2 = ingest_with_asset(asset_b, 1)

# Read back items via the STAC items list endpoint.  The list endpoint
# only supports limit/offset/lang — to filter by external_id you would
# use the platform `/search/catalogs/{cat}` POST surface
# with a CQL2 filter; we do the simpler list + grep here.
list_url = f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"
r = client.get(list_url, params={"limit": 50})
rows = r.json().get("features", []) if r.status_code == 200 else []
k42_rows = [row for row in rows if row.get("properties", {}).get("code") == "K42"]
print(f"\nK42 versions found: {len(k42_rows)}")
for row in k42_rows:
    props = row.get("properties", {})
    print(f"  geoid={row.get('id')} asset_id={props.get('asset_id')!r} "
          f"valid_from={props.get('valid_from')} valid_to={props.get('valid_to')}")


## Teardown

In [ ]:
# Teardown — delete the ephemeral catalog. Comment out to keep state.
r = client.delete(
    f"/stac/catalogs/{CATALOG_ID}",
    params={"force": "true"},
    timeout=60.0,
)
print(f"teardown DELETE /stac/catalogs/{CATALOG_ID}: {r.status_code}")
client.close()
